# INF201 – Deliverable 4

Task 1: a `Temperature` class (Celsius, Fahrenheit, Kelvin).
Task 2: a `Fraction` class.

Each task has the class, one section per requirement with a short demonstration, and finally the output from the pytest test suite (`test_temperature.py` and `test_fraction.py`).

# Task 1 – Temperatures

## 1.0 The `Temperature` class

Values are stored as given; all comparisons are done in Kelvin with a tolerance of 1e-9. `value` and `unit` are read-only properties.

In [25]:
"""Temperature class supporting Celsius, Fahrenheit and Kelvin."""

import math

TOLERANCE = 1e-9
_VALID_UNITS = ("C", "F", "K")


class Temperature:
    """A temperature in Celsius ("C"), Fahrenheit ("F") or Kelvin ("K").

    Temperatures can be converted between units and compared across units.
    Instances are immutable: ``value`` and ``unit`` are read-only.
    """

    def __init__(self, value, unit="C"):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"value must be a number, got {type(value).__name__}")
        if math.isnan(value):
            raise ValueError("value must not be NaN")
        if not isinstance(unit, str) or unit.upper() not in _VALID_UNITS:
            raise ValueError(f"unit must be one of 'C', 'F' or 'K', got {unit!r}")

        unit = unit.upper()
        kelvin = self._to_kelvin_static(value, unit)
        if kelvin < -TOLERANCE:
            raise ValueError(
                f"temperature {value} {unit} is below absolute zero (0 K)"
            )

        self._value = float(value)
        self._unit = unit

    # ------------------------------------------------------------------
    # Read-only attributes
    # ------------------------------------------------------------------
    @property
    def value(self) -> float:
        return self._value

    @property
    def unit(self) -> str:
        return self._unit

    # ------------------------------------------------------------------
    # Conversions
    # ------------------------------------------------------------------
    @staticmethod
    def _to_kelvin_static(value, unit):
        if unit == "K":
            return value
        if unit == "C":
            return value + 273.15
        return (value - 32) * 5 / 9 + 273.15  # "F"

    def to_celsius(self) -> float:
        if self._unit == "C":
            return self._value
        if self._unit == "K":
            return self._value - 273.15
        return (self._value - 32) * 5 / 9

    def to_fahrenheit(self) -> float:
        if self._unit == "F":
            return self._value
        if self._unit == "C":
            return self._value * 9 / 5 + 32
        return (self._value - 273.15) * 9 / 5 + 32  # "K"

    def to_kelvin(self) -> float:
        if self._unit == "K":
            return self._value
        if self._unit == "C":
            return self._value + 273.15
        return (self._value - 32) * 5 / 9 + 273.15  # "F"

    def convert_to(self, target_unit: str) -> "Temperature":
        """Return a new Temperature expressed in ``target_unit``."""
        if not isinstance(target_unit, str) or target_unit.upper() not in _VALID_UNITS:
            raise ValueError(
                f"target_unit must be one of 'C', 'F' or 'K', got {target_unit!r}"
            )
        target = target_unit.upper()
        if target == "C":
            return Temperature(self.to_celsius(), "C")
        if target == "F":
            return Temperature(self.to_fahrenheit(), "F")
        return Temperature(self.to_kelvin(), "K")

    # ------------------------------------------------------------------
    # String representations
    # ------------------------------------------------------------------
    def __repr__(self) -> str:
        return f"Temperature({self._value!r}, {self._unit!r})"

    def __str__(self) -> str:
        return f"{self._value:.2f} °{self._unit}" if self._unit != "K" else f"{self._value:.2f} K"

    # ------------------------------------------------------------------
    # Comparisons (done in Kelvin, with a tolerance of 1e-9)
    # ------------------------------------------------------------------
    def _is_close(self, other: "Temperature") -> bool:
        return abs(self.to_kelvin() - other.to_kelvin()) <= TOLERANCE

    def __eq__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self._is_close(other)

    def __ne__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return not self._is_close(other)

    def __lt__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return not self._is_close(other) and self.to_kelvin() < other.to_kelvin()

    def __le__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self._is_close(other) or self.to_kelvin() < other.to_kelvin()

    def __gt__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return not self._is_close(other) and self.to_kelvin() > other.to_kelvin()

    def __ge__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self._is_close(other) or self.to_kelvin() > other.to_kelvin()

    # Equality uses a tolerance, so a consistent hash is not possible.
    __hash__ = None

## 1.1 Initialisation

**Default unit, case-insensitive units and read-only attributes**

In [26]:
print(repr(Temperature(25)))            # default unit is Celsius
print(repr(Temperature(300, "k")))      # lower case is accepted
print(repr(Temperature(98.6, "f")))

t = Temperature(25, "C")
for attr, new in (("value", 30), ("unit", "F")):
    try:
        setattr(t, attr, new)
    except AttributeError as e:
        print(f"Setting {attr!r} failed:", e)

Temperature(25.0, 'C')
Temperature(300.0, 'K')
Temperature(98.6, 'F')
Setting 'value' failed: property 'value' of 'Temperature' object has no setter
Setting 'unit' failed: property 'unit' of 'Temperature' object has no setter


**Result:** the unit is normalised to upper case, defaults to `C`, and both attributes are read-only.

## 1.2 Validation

**Invalid units and temperatures below absolute zero**

In [27]:
for value, unit in [(10, "X"), (10, ""), (10, "Celsius"), (-273.16, "C"), (-459.68, "F"), (-0.001, "K")]:
    try:
        Temperature(value, unit)
        print(f"Temperature({value}, {unit!r}) -> accepted")
    except ValueError as e:
        print(f"Temperature({value}, {unit!r}) -> ValueError: {e}")

print()
for value, unit in [(-273.15, "C"), (-459.67, "F"), (0, "K")]:
    print("Absolute zero is allowed:", repr(Temperature(value, unit)))

Temperature(10, 'X') -> ValueError: unit must be one of 'C', 'F' or 'K', got 'X'
Temperature(10, '') -> ValueError: unit must be one of 'C', 'F' or 'K', got ''
Temperature(10, 'Celsius') -> ValueError: unit must be one of 'C', 'F' or 'K', got 'Celsius'
Temperature(-273.16, 'C') -> ValueError: temperature -273.16 C is below absolute zero (0 K)
Temperature(-459.68, 'F') -> ValueError: temperature -459.68 F is below absolute zero (0 K)
Temperature(-0.001, 'K') -> ValueError: temperature -0.001 K is below absolute zero (0 K)

Absolute zero is allowed: Temperature(-273.15, 'C')
Absolute zero is allowed: Temperature(-459.67, 'F')
Absolute zero is allowed: Temperature(0.0, 'K')


## 1.3 Conversions

**Accuracy at the reference points** (absolute zero, C/F crossover, freezing, body temperature, boiling)

In [28]:
points = [("Absolute zero", -273.15, -459.67, 0.0),
          ("C/F crossover", -40.0, -40.0, 233.15),
          ("Freezing water", 0.0, 32.0, 273.15),
          ("Human body", 37.0, 98.6, 310.15),
          ("Boiling water", 100.0, 212.0, 373.15)]

for name, c, f, k in points:
    ok = all(abs(x - y) < 1e-9 for x, y in [
        (Temperature(c, "C").to_fahrenheit(), f), (Temperature(c, "C").to_kelvin(), k),
        (Temperature(f, "F").to_celsius(), c),    (Temperature(f, "F").to_kelvin(), k),
        (Temperature(k, "K").to_celsius(), c),    (Temperature(k, "K").to_fahrenheit(), f)])
    print(f"{name:15s} {c:8.2f} C = {f:8.2f} F = {k:7.2f} K   all six conversions correct: {ok}")

print()
print(repr(Temperature(100, "C").convert_to("f")))
try:
    Temperature(100, "C").convert_to("X")
except ValueError as e:
    print("convert_to('X') -> ValueError:", e)

Absolute zero    -273.15 C =  -459.67 F =    0.00 K   all six conversions correct: True
C/F crossover     -40.00 C =   -40.00 F =  233.15 K   all six conversions correct: True
Freezing water      0.00 C =    32.00 F =  273.15 K   all six conversions correct: True
Human body         37.00 C =    98.60 F =  310.15 K   all six conversions correct: True
Boiling water     100.00 C =   212.00 F =  373.15 K   all six conversions correct: True

Temperature(212.0, 'F')
convert_to('X') -> ValueError: target_unit must be one of 'C', 'F' or 'K', got 'X'


## 1.4 Round-trip conversion

**C → F → K → C reproduces the original temperature**

In [29]:
for c in [-273.15, -40, 0, 37, 100, 1234.5]:
    original = Temperature(c, "C")
    back = original.convert_to("F").convert_to("K").convert_to("C")
    print(f"{original!s:>12}  ->  {back!s:>12}   equal: {original == back}")

  -273.15 °C  ->    -273.15 °C   equal: True
   -40.00 °C  ->     -40.00 °C   equal: True
     0.00 °C  ->       0.00 °C   equal: True
    37.00 °C  ->      37.00 °C   equal: True
   100.00 °C  ->     100.00 °C   equal: True
  1234.50 °C  ->    1234.50 °C   equal: True


## 1.5 Cross-unit comparisons

In [30]:
print("0 C == 32 F :", Temperature(0, "C") == Temperature(32, "F"))
print("0 C != 32 F :", Temperature(0, "C") != Temperature(32, "F"))

cold, warm, hot = Temperature(0, "C"), Temperature(100, "F"), Temperature(400, "K")
print("0 C < 100 F < 400 K   :", cold < warm < hot)
print("0 C <= 100 F <= 400 K :", cold <= warm <= hot)
print("400 K > 100 F > 0 C   :", hot > warm > cold)
print("400 K >= 100 F >= 0 C :", hot >= warm >= cold)

print()
print("Tolerance 1e-9:", Temperature(25.0, "C") == Temperature(25.0 + 1e-10, "C"),
      Temperature(25.0, "C") == Temperature(25.0 + 1e-6, "C"))
print("Compare with int, __eq__ returns:", Temperature(10, "C").__eq__(10))
print("Temperature(10, 'C') == 10 ->", Temperature(10, "C") == 10)

0 C == 32 F : True
0 C != 32 F : False
0 C < 100 F < 400 K   : True
0 C <= 100 F <= 400 K : True
400 K > 100 F > 0 C   : True
400 K >= 100 F >= 0 C : True

Tolerance 1e-9: True False
Compare with int, __eq__ returns: NotImplemented
Temperature(10, 'C') == 10 -> False


**Result:** comparisons work across units, use the 1e-9 tolerance and return `NotImplemented` for non-`Temperature` objects.

## 1.6 Formatting

In [31]:
print(str(Temperature(25, "C")))
print(str(Temperature(98.6, "F")))
print(str(Temperature(310.15, "K")))
print(str(Temperature(25.126, "C")))
print(repr(Temperature(25, "C")))

25.00 °C
98.60 °F
310.15 K
25.13 °C
Temperature(25.0, 'C')


## 1.7 Test output – `test_temperature.py`

In [32]:
!python -m pytest test_temperature.py -v -p no:cacheprovider

============================= test session starts =============================
platform win32 -- Python 3.13.14, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\risha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
rootdir: c:\Users\risha\Downloads\INF201\Week_48
collecting ... collected 79 items

test_temperature.py::TestInitialization::test_default_unit_is_celsius PASSED [  1%]
test_temperature.py::TestInitialization::test_value_is_stored_as_float PASSED [  2%]
test_temperature.py::TestInitialization::test_case_insensitive_units[c] PASSED [  3%]
test_temperature.py::TestInitialization::test_case_insensitive_units[C] PASSED [  5%]
test_temperature.py::TestInitialization::test_case_insensitive_units[f] PASSED [  6%]
test_temperature.py::TestInitialization::test_case_insensitive_units[F] PASSED [  7%]
test_temperature.py::TestInitialization::test_case_insensitive_units[k] PASSED [  8%]
test_temperature.py::TestInitialization::test_case_insensiti